# 8 — reading the trace

Since book 1, `configure_tracing` has sent a record of every model call,
tool call and agent run to LangSmith, one project per book —
`hotelbot-book-1` through `hotelbot-book-7`. No book has looked at that
record properly yet. This one runs a single booking conversation from
first question to confirmed reservation, then reads it back from the other
side: what happened in what order, how many tokens each step used, what
that cost, and where the time went. First in the LangSmith web page, then
the same numbers pulled into Python.

Two words carry the whole book. A **run** is one recorded step: one model
call, one tool call, or one call of the whole agent, with its inputs,
outputs, start time and end time. Runs nest — the agent's run contains the
model runs and tool runs it made along the way. A **trace** is one
top-level run together with everything nested inside it.

This notebook needs `LANGSMITH_API_KEY` in `.env`. The earlier books
worked without it; this one reads the record back, so there has to be one.

In [1]:
import os
import warnings

from langchain.agents.middleware import ToolRetryMiddleware
from langsmith import Client

from hotelbot.agent import build_agent, resume, run
from hotelbot.config import configure_tracing
from hotelbot.middleware import ScreenRetrievedText
from hotelbot.reservations import reset_reservations

configure_tracing(book=8)
if not os.environ.get("LANGSMITH_API_KEY"):
    raise RuntimeError("Book 8 reads traces back from LangSmith: add LANGSMITH_API_KEY to .env and restart the kernel.")

# list_runs() still works but prints a deprecation notice on every call.
warnings.filterwarnings("ignore", message=r"list_runs\(\) is deprecated")

PROJECT = "hotelbot-book-8"
reset_reservations()
agent = build_agent(
    middleware=[ToolRetryMiddleware(max_retries=2, tools=["check_availability"]), ScreenRetrievedText()],
)
client = Client()

**Labelling a run.** `run()` and `resume()` take a `config` dict that is
handed to the agent along with the thread id. Three keys in it end up in
LangSmith:

- `tags` — a list of short strings. Many runs can share a tag, and you can
  ask LangSmith for every run that has one.
- `metadata` — a dict of names and values, here `{"guest": "olek"}`.
  Searchable the same way.
- `run_id` — the id the top run of this call will get. LangSmith normally
  makes one up; choosing it yourself means the notebook knows where to find
  the trace afterwards without searching for it.

Each call of `run()` or `resume()` is one call of the agent, so each one
makes its own trace. `labelled()` gives every turn the same tags and
metadata and a fresh run id, and keeps the ids in `trace_ids`, in order.
`show()` prints the agent's typed answer, or what it paused on.

The first turn: a city, dates and a budget.

In [2]:
import uuid

thread_id = str(uuid.uuid4())
trace_ids = []


def labelled():
    trace_ids.append(uuid.uuid4())
    return {"tags": ["book-8", "full-booking"], "metadata": {"guest": "olek"}, "run_id": trace_ids[-1]}


def show(result):
    if result.interrupt:
        request = result.interrupt["action_requests"][0]
        print("paused before:", request["name"], request["args"])
    else:
        print(result.structured_response)


result = run(agent, "I need two nights in Lisbon, 2026-10-10 to 2026-10-12, under €150 a night.", thread_id, labelled())
show(result)

text='Here are options in Lisbon for Oct 10–12 (2 nights), all under €150/night:\n\n1. Alfama House – €90/night (€180 total) – wifi, gym, pool, 4.5★\n2. Alfama Loft – €125/night (€250 total) – wifi, breakfast, 4.1★\n3. Belem Suites – €134/night (€268 total) – pool, parking, spa, 4.8★\n4. Casa do Castelo – €140/night (€280 total) – breakfast, wifi, gym, 4.6★\n\nWant me to book one?'


**Narrowing, then a policy question.** Same thread, two more turns. The
second asks for breakfast and for the dates to be checked, so the agent
calls `check_availability`. The third is about cancelling, which only the
policy documents answer, so it calls `lookup_policy` — and
`ScreenRetrievedText` from book 7 sends what comes back through a Haiku
call before Sonnet reads it.

In [3]:
for text in [
    "Only the ones with breakfast included — and check they're actually free on those dates.",
    "If I book and then have to cancel a week before check-in, what do I lose?",
]:
    print(">", text)
    show(run(agent, text, thread_id, labelled()))
    print()

> Only the ones with breakfast included — and check they're actually free on those dates.
text='Both breakfast-included options are confirmed available for Oct 10–12:\n\n1. Alfama Loft – €125/night, €250 total – wifi, breakfast, 4.1★\n2. Casa do Castelo – €140/night, €280 total – breakfast, wifi, gym, 4.6★\n\nWant me to book one?'

> If I book and then have to cancel a week before check-in, what do I lose?
text="Cancelling a week out is outside the 5-day window, so it's free — no charge, full refund."



**Book it.** Asking for the booking makes the agent call
`make_reservation`, and the approval gate from book 5 pauses the run before
that call happens. This turn's trace ends at the pause.

In [4]:
result = run(agent, "Book the cheaper one under the name Olek Pedenko.", thread_id, labelled())
show(result)

paused before: make_reservation {'hotel_id': 'lis-002', 'guest': 'Olek Pedenko', 'check_in': '2026-10-10', 'check_out': '2026-10-12'}


**Approve.** `resume()` continues the paused run with your decision, as a
new call of the agent — so a fifth trace. Wait a little before running this
cell; that wait comes back later, when the time is counted.

In [5]:
result = resume(agent, {"type": "approve"}, thread_id, labelled())
show(result)

text='Booked! Alfama Loft, Oct 10–12, 2 nights, €250 total, under Olek Pedenko. Confirmation ID: res-0001.'


**Where the five traces are.** Runs aren't uploaded while the agent works.
They're queued and sent in the background, a batch at a time, so a cell can
finish before its runs have left your machine. `wait_for_all_tracers()`
blocks until everything queued so far has been sent.

Once sent, LangSmith takes a moment more to file them. `read_back()` asks
for a run by id with `client.read_run()`, and if LangSmith doesn't have it
yet, waits a second and asks again. `client.get_run_url()` turns a run into
the address of its page.

In [6]:
import time

from langchain_core.tracers.langchain import wait_for_all_tracers
from langsmith.utils import LangSmithNotFoundError


def read_back(run_id):
    for _ in range(15):
        try:
            return client.read_run(run_id)
        except LangSmithNotFoundError:
            time.sleep(1)
    raise LangSmithNotFoundError(f"run {run_id} still not in LangSmith after 15 seconds")


wait_for_all_tracers()
tops = [read_back(trace_id) for trace_id in trace_ids]
for turn, top in enumerate(tops, start=1):
    print(f"turn {turn}: {client.get_run_url(run=top)}")

/var/folders/v0/wygnm4fn6bg8lydkvjt_2xmh0000gn/T/ipykernel_35453/2566088520.py:10: DeprecationWarning: read_run() is deprecated and will be removed after Jan 31, 2027. Use client.runs.retrieve() instead. See https://docs.langchain.com/langsmith/smithdb-sdk-migration-runs#runs-retrieve for the migration guide.
  return client.read_run(run_id)
/var/folders/v0/wygnm4fn6bg8lydkvjt_2xmh0000gn/T/ipykernel_35453/2566088520.py:19: DeprecationWarning: get_run_url() is deprecated and will be removed after Jan 31, 2027. Use client.runs.get_url() instead. See https://docs.langchain.com/langsmith/smithdb-sdk-migration-runs#runs-get-url for the migration guide.
  print(f"turn {turn}: {client.get_run_url(run=top)}")


turn 1: https://smith.langchain.com/o/d590007d-a6a0-46e5-9a56-baddd100e9d2/projects/p/fe3bae13-1aac-48d2-8cb3-7f1bc4f16095/r/b681b7c3-cd13-4c1f-84a2-c7d628a5364f?poll=true
turn 2: https://smith.langchain.com/o/d590007d-a6a0-46e5-9a56-baddd100e9d2/projects/p/fe3bae13-1aac-48d2-8cb3-7f1bc4f16095/r/f6ec8bdc-3116-4603-aa31-b6b92661c54a?poll=true
turn 3: https://smith.langchain.com/o/d590007d-a6a0-46e5-9a56-baddd100e9d2/projects/p/fe3bae13-1aac-48d2-8cb3-7f1bc4f16095/r/723f14f3-5ec0-439f-87db-b47973781f29?poll=true
turn 4: https://smith.langchain.com/o/d590007d-a6a0-46e5-9a56-baddd100e9d2/projects/p/fe3bae13-1aac-48d2-8cb3-7f1bc4f16095/r/073eeb7c-a0c1-4159-90f2-2b36869faf7a?poll=true
turn 5: https://smith.langchain.com/o/d590007d-a6a0-46e5-9a56-baddd100e9d2/projects/p/fe3bae13-1aac-48d2-8cb3-7f1bc4f16095/r/adb8ad4d-6b12-49d4-a984-7ba8197bb8a6?poll=true


**The tree.** Open turn 2's link. The left side of the page is the trace as
a tree: each row is a run, and a row indented under another ran inside it.
The top row is the whole agent call. Under it, the steps of the loop in the
order they happened: a model step holding the call to Sonnet, a tools step
holding the tool calls that model call asked for, then a model step again,
until the last model call writes the typed answer.

What to find:

- **Your own function in the tree.** Inside the `check_availability` tool
  row sits a row called `check_availability.python`. That's the plain
  Python function in `hotelbot/catalogue.py` — the tool is a thin wrapper
  around it. It shows up because the function now has
  `@traceable(name="check_availability.python")` above it: `traceable`
  records each call of the function as a run, nested under whatever run was
  active when it was called. Without an API key it just calls the function.
- **The screen.** In turn 3, find the `lookup_policy` row, and a model call
  near it in the same tools step that is not Sonnet: the Haiku call that
  judged the retrieved text. Click it to see the policy text it was shown.
- **The pause.** Turn 4's tree ends with a model call that asked for
  `make_reservation` — and no `make_reservation` row, because the gate
  stopped the run first. Turn 5 starts with that row. Inside it,
  `check_availability.python` appears again: writing a reservation checks
  availability through the same function.
- **The conversation.** The project's Threads tab groups traces by the
  `thread_id` in their metadata, so all five turns show up there as one
  conversation.

The same tree is available from Python. `client.list_runs(trace_id=...)`
returns every run in one trace. Each run knows its `parent_run_id` — the
run it sits inside — so grouping runs by parent and printing children
under their parent, indented one step further, rebuilds the tree.

In [7]:
from collections import defaultdict


def print_tree(trace_id):
    children = defaultdict(list)
    for r in client.list_runs(project_name=PROJECT, trace_id=trace_id):
        children[r.parent_run_id].append(r)

    def walk(parent_id, depth):
        for r in sorted(children[parent_id], key=lambda r: r.start_time):
            print("    " * depth + f"{r.name}  [{r.run_type}]")
            walk(r.id, depth + 1)

    walk(None, 0)


print_tree(trace_ids[1])

LangGraph  [chain]
    model  [chain]
        ChatAnthropic  [llm]
    HumanInTheLoopMiddleware.after_model  [chain]
    tools  [chain]
        ToolRetryMiddleware.wrap_tool_call  [chain]
            ScreenRetrievedText.wrap_tool_call  [chain]
                check_availability  [tool]
                    check_availability.python  [chain]
    tools  [chain]
        ToolRetryMiddleware.wrap_tool_call  [chain]
            ScreenRetrievedText.wrap_tool_call  [chain]
                check_availability  [tool]
                    check_availability.python  [chain]
    model  [chain]
        ChatAnthropic  [llm]
    HumanInTheLoopMiddleware.after_model  [chain]


**Tokens per model call.** A token is the unit models read and write in —
roughly a short word or a piece of a longer one. Every model run records
two counts: `prompt_tokens`, everything the model was sent (system prompt,
tool descriptions, the conversation so far), and `completion_tokens`,
everything it wrote back. `total_tokens` is the two added together.

`run_type="llm"` narrows `list_runs` to model calls. Each one also records
which model answered, under `ls_model_name` in its metadata.

Before running: the model calls are listed in the order they happened.
Will `in` (prompt tokens) go up, go down, or jump around from one call to
the next?

In [8]:
def model_name(r):
    return r.extra.get("metadata", {}).get("ls_model_name", "?")


model_runs = []  # (turn, run)
for turn, trace_id in enumerate(trace_ids, start=1):
    for r in client.list_runs(project_name=PROJECT, trace_id=trace_id, run_type="llm"):
        model_runs.append((turn, r))
model_runs.sort(key=lambda pair: pair[1].start_time)

for turn, r in model_runs:
    print(f"turn {turn}  {model_name(r):27} in {r.prompt_tokens:6}  out {r.completion_tokens:5}  total {r.total_tokens:6}")

turn 1  claude-sonnet-5             in   2507  out    66  total   2573
turn 1  claude-sonnet-5             in   3081  out   226  total   3307
turn 2  claude-sonnet-5             in   3553  out   209  total   3762
turn 2  claude-sonnet-5             in   3886  out   146  total   4032
turn 3  claude-sonnet-5             in   4195  out    60  total   4255
turn 3  claude-haiku-4-5-20251001   in   1116  out    92  total   1208
turn 3  claude-sonnet-5             in   4707  out    71  total   4778
turn 4  claude-sonnet-5             in   4855  out   131  total   4986
turn 5  claude-sonnet-5             in   5099  out    94  total   5193


**The prompt is resent every time.** A model keeps nothing between calls.
Each call is sent the whole conversation again — every earlier question,
answer, tool call and tool result — plus the system prompt and the tool
descriptions. So the Sonnet `in` column climbs call after call, and it
climbs most right after a tool returned something long: three passages of
policy text arrive as a tool result, and from then on every call carries
them.

Added up per turn, below. The turns late in the conversation pay for
everything that came before them.

In [9]:
per_turn = defaultdict(int)
for turn, r in model_runs:
    per_turn[turn] += r.total_tokens

for turn, tokens in sorted(per_turn.items()):
    print(f"turn {turn}: {tokens:7} tokens")

turn 1:    5880 tokens
turn 2:    7794 tokens
turn 3:   10241 tokens
turn 4:    4986 tokens
turn 5:    5193 tokens


**Two sources, one number.** Book 1 showed `usage_metadata` on an
`AIMessage`: the token counts the provider reported for the call that wrote
it. The thread's saved messages include every `AIMessage` Sonnet wrote in
this conversation, so their counts added up should match the Sonnet runs
in the trace.

Before running: the trace also has Haiku rows. Will those be in the
messages' total too?

In [10]:
from langchain_core.messages import AIMessage

from_messages = sum(m.usage_metadata["total_tokens"] for m in result.messages if isinstance(m, AIMessage))

from_trace = defaultdict(int)
for _, r in model_runs:
    from_trace[model_name(r)] += r.total_tokens

print("AIMessages on the thread:", from_messages)
for name, tokens in from_trace.items():
    print(f"trace, {name}: {tokens}")

AIMessages on the thread: 32886
trace, claude-sonnet-5: 32886
trace, claude-haiku-4-5-20251001: 1208


The Sonnet line matches the messages. The Haiku tokens appear only in the
trace: the screen's call happens inside the tools step, and its answer is
used to decide what the tool returns — it never becomes a message in the
conversation. The messages show what the model said; the trace shows
everything that ran.

**From tokens to dollars.** Anthropic charges per million tokens, at one
price for tokens sent in and a higher one for tokens written out.
`PRICING` in `hotelbot/config.py` holds the two prices for each model, in
US dollars, copied from the price list on the day it was written — prices
change, and these don't follow them.

Sonnet 5 is (2.00, 10.00): $2 per million tokens in, $10 per million out.
So a call with 10,000 tokens in and 500 out costs

- in: 10,000 / 1,000,000 × 2.00 = $0.020
- out: 500 / 1,000,000 × 10.00 = $0.005
- total: $0.025

`cost()` does exactly that for one run, using that run's model.

In [11]:
from hotelbot.config import PRICING


def cost(r):
    price_in, price_out = PRICING[model_name(r)]
    return (r.prompt_tokens * price_in + r.completion_tokens * price_out) / 1_000_000


booking_cost = sum(cost(r) for _, r in model_runs)
for name in from_trace:
    print(f"{name}: ${sum(cost(r) for _, r in model_runs if model_name(r) == name):.4f}")
print(f"whole booking: ${booking_cost:.4f}")

claude-sonnet-5: $0.0738
claude-haiku-4-5-20251001: $0.0016
whole booking: $0.0754


**Against one plain question.** Book 1's first cell asked Sonnet "Name
three cities in Portugal." — one model call, no agent, no tools, no system
prompt. The cell below asks it again the same way, with a `run_id` in the
config so its run can be read back like the turns were. Called directly
like this, the model call is a trace on its own.

Predict before running: how many times more did the booking cost than this
one question? 5×? 50×? 500×?

In [12]:
from langchain.chat_models import init_chat_model

from hotelbot.config import CHAT_MODEL

question_id = uuid.uuid4()
init_chat_model(CHAT_MODEL, reasoning_effort="low").invoke(
    "Name three cities in Portugal.", config={"tags": ["book-8"], "run_id": question_id},
)
wait_for_all_tracers()
question = read_back(question_id)

print(f"one question: {question.total_tokens:6} tokens  ${cost(question):.5f}")
print(f"one booking:  {sum(per_turn.values()):6} tokens  ${booking_cost:.5f}")
print(f"ratio: {booking_cost / cost(question):.0f}×")

/var/folders/v0/wygnm4fn6bg8lydkvjt_2xmh0000gn/T/ipykernel_35453/2566088520.py:10: DeprecationWarning: read_run() is deprecated and will be removed after Jan 31, 2027. Use client.runs.retrieve() instead. See https://docs.langchain.com/langsmith/smithdb-sdk-migration-runs#runs-retrieve for the migration guide.
  return client.read_run(run_id)


one question:     89 tokens  $0.00075
one booking:   34094 tokens  $0.07537
ratio: 100×


**Time per run.** Each run has `start_time` and `end_time`. Subtracting
them gives a `timedelta`, and `.total_seconds()` turns that into a plain
number of seconds.

For each turn: how long the whole agent call took, how much of that was
spent inside model calls, and how much inside tool calls. Predict before
running: roughly what share of each turn is model time?

In [13]:
def seconds(runs):
    return sum((r.end_time - r.start_time).total_seconds() for r in runs)


print("turn   whole   models   tools")
for turn, (trace_id, top) in enumerate(zip(trace_ids, tops), start=1):
    runs = list(client.list_runs(project_name=PROJECT, trace_id=trace_id))
    models = [r for r in runs if r.run_type == "llm"]
    tools = [r for r in runs if r.run_type == "tool"]
    print(f"{turn:4}  {seconds([top]):6.1f}  {seconds(models):7.1f}  {seconds(tools):6.2f}")

turn   whole   models   tools
   1     3.6      3.5    0.00
   2     3.7      3.7    0.01
   3     6.5      3.9    2.53
   4     1.7      1.7    0.00
   5     1.4      1.4    0.00


Model calls take seconds; the catalogue tools take milliseconds, and
`lookup_policy` is slower only because it calls the embeddings API and
then Haiku. A turn lasts about as long as its model calls added together.
Two things bend that: the Haiku screen is a model call *inside* a tool, so
its seconds count in both columns; and when the model asks for several
tools at once they run side by side, so their times can add up to more
than the time they actually took.

**The time nobody recorded.** The pause between turn 4 and turn 5 is in no
run at all — turn 4's trace ended when the gate stopped it, and turn 5's
started when you approved. The gap between the two is how long the booking
waited for you.

In [14]:
waited = (tops[4].start_time - tops[3].end_time).total_seconds()
print(f"agent working, all five turns: {seconds(tops):.1f} s")
print(f"waiting for your approval:     {waited:.1f} s")

agent working, all five turns: 16.9 s
waiting for your approval:     0.0 s


**Finding runs by label.** `list_runs` takes a `filter`: a small
expression LangSmith checks against every run, returning those it's true
for.

- `has(tags, "full-booking")` — true when the run's tags include
  `"full-booking"`.
- `and(eq(metadata_key, "guest"), eq(metadata_value, "olek"))` — true when
  the run's metadata has a `guest` whose value is `"olek"`.

Tags and metadata passed to the agent are copied onto every run inside it,
so `is_root=True` is added to count each turn once. Each run of this
notebook adds five tagged traces; the `thread_id` in their metadata tells
the conversations apart.

In [15]:
tagged = list(client.list_runs(project_name=PROJECT, is_root=True, filter='has(tags, "full-booking")'))
conversations = {r.extra.get("metadata", {}).get("thread_id") for r in tagged}
print(f'has(tags, "full-booking"): {len(tagged)} turns in {len(conversations)} conversation(s)')

olek = list(client.list_runs(
    project_name=PROJECT, is_root=True, filter='and(eq(metadata_key, "guest"), eq(metadata_value, "olek"))',
))
print(f"guest = olek:              {len(olek)} turns")

has(tags, "full-booking"): 5 turns in 1 conversation(s)
guest = olek:              5 turns


**What each book cost.** The same three steps — every model run in a
project, its tokens, `cost()` — across all eight projects. A project that
doesn't exist (a book run without a key) gets a line saying so. A model
missing from `PRICING` is counted in tokens but not in dollars, and the
`unpriced` column says how many such calls there were.

LangSmith doesn't keep runs forever. On the free plan they're deleted after
about two weeks, while the project itself stays — so the oldest books show
up with zero model calls. The table counts what is left, not everything
that ever ran.

Before running: of the books that still have runs, which was the most
expensive to write?

In [16]:
print("book  model calls     tokens   cost $  unpriced")
for book in range(1, 9):
    try:
        runs = list(client.list_runs(project_name=f"hotelbot-book-{book}", run_type="llm"))
    except LangSmithNotFoundError:
        print(f"{book:4}  no project")
        continue
    priced = [r for r in runs if model_name(r) in PRICING and r.total_tokens]
    tokens = sum(r.total_tokens or 0 for r in runs)
    dollars = sum(cost(r) for r in priced)
    print(f"{book:4}  {len(runs):11}  {tokens:9}  {dollars:7.3f}  {len(runs) - len(priced):8}")

book  model calls     tokens   cost $  unpriced
   1            0          0    0.000         0
   2          106     174377    0.439         4
   3           79     207586    0.478         0
   4           14      37786    0.087         0
   5          168     567173    1.282         1
   6          139     442755    1.017         0
   7           63     140184    0.329        14
   8           10      34183    0.076         0


**What you can do now.** A conversation with the agent is no longer
something you only see the end of. Every model call, tool call and pause
is in the trace, nested the way it ran, and your own Python functions can
join it with one decorator. Tokens, dollars and seconds come back with a
few lines of `list_runs`, for one conversation or a whole project — and
the trace counts calls, like the screen's, that the conversation itself
never shows.

What the trace can't tell you is whether any of it was right. Every number
here would look the same if the agent had proposed the wrong hotel or
misread the cancellation policy. Saying whether an answer was good needs
expected answers to compare against and a way to score them — evaluation,
which is where the next project starts.